# BestShot — Full Benchmark Pipeline

Run this notebook from **Experiment > Jupyter Interface** on chameleoncloud.org.

**Before running:**
1. You must have an active `gpu_mi100` lease on CHI@TACC
2. Your SSH key must be uploaded to CHI@TACC
3. The repo must already be cloned on the node (or will be cloned in this notebook)

## 1. Connect to Chameleon project

In [ ]:
from chi import server, context, lease
import os
import time

context.version = "1.0"
context.choose_project()
context.choose_site(default="CHI@TACC")

## 2. Get your active lease

In [ ]:
LEASE_NAME = "serve_proj19_lease"   # <-- your lease name

l = lease.get_lease(LEASE_NAME)
l.show()   # should say ACTIVE

## 3. Connect to existing server

If the server is already running from a previous session, just reconnect to it.
If not, run the full setup notebook first.

In [ ]:
username = os.getenv("USER")
s = server.Server(f"node-bestshot-{username}")
s.refresh()
s.show(type="widget")   # note the floating IP

## 4. Pull latest code and rebuild Docker image

In [ ]:
print("Pulling latest code...")
s.execute("cd bestshot && git pull")

In [ ]:
print("Building Docker image (this may take a few minutes)...")
s.execute("docker build -t bestshot-serve -f bestshot/serving/docker/Dockerfile.serve bestshot/serving/")
print("Build complete!")

## 5. Helper functions

In [ ]:
def start_server(config_name, gpu=False):
    """Start the FastAPI server with the given config."""
    print(f"\nStarting server with config: {config_name}")
    
    # clean up any old container
    s.execute("docker rm -f bestshot-api 2>/dev/null || true")
    
    gpu_flags = "--device=/dev/kfd --device=/dev/dri --group-add video" if gpu else ""
    
    s.execute(
        f"docker run -d --network host "
        f"{gpu_flags} "
        f"--name bestshot-api "
        f"-e CONFIG_NAME={config_name} "
        f"bestshot-serve "
        f"uvicorn app:app --host 0.0.0.0 --port 8000"
    )
    
    # wait for server to be ready
    print("Waiting for server to be ready...")
    s.execute(
        "until curl -s http://127.0.0.1:8000/docs > /dev/null; do "
        "sleep 2; echo 'still waiting...'; done && echo 'Server ready!'"
    )

def run_benchmark(config_name, gpu=False):
    """Run benchmark against the running server."""
    print(f"\n Starting benchmark: {config_name}")
    start = time.time()
    
    gpu_flags = "--device=/dev/kfd --device=/dev/dri --group-add video" if gpu else ""
    
    s.execute(
        f"docker run --rm --network host "
        f"{gpu_flags} "
        f"-e CONFIG_NAME={config_name} "
        f"bestshot-serve "
        f"sh -c 'python benchmark.py'"
    )
    
    elapsed = time.time() - start
    print(f"Done: {config_name} — took {elapsed:.1f}s total")
    
def stop_server():
    """Stop the API server."""
    s.execute("docker stop bestshot-api 2>/dev/null || true")
    print("Server stopped.")

def check_resource_usage():
    """Snapshot of CPU and memory usage."""
    s.execute(
        "docker stats bestshot-api --no-stream "
        "--format 'CPU: {{.CPUPerc}}  MEM: {{.MemUsage}}'"
    )

print("Helper functions loaded.")

---
## 6. Config 1 — CPU Sequential (baseline)

In [ ]:
start_server("cpu_sequential", gpu=False)

In [ ]:
check_resource_usage()

In [ ]:
run_benchmark("cpu_sequential", gpu=False)

In [ ]:
stop_server()

---
## 7. Config 2 — CPU Batch (system-level: concurrency)

In [ ]:
start_server("cpu_batch", gpu=False)

In [ ]:
check_resource_usage()

In [ ]:
run_benchmark("cpu_batch", gpu=False)

In [ ]:
stop_server()

---
## 8. Config 3 — GPU Sequential (infrastructure-level: AMD MI100)

In [ ]:
start_server("gpu_sequential", gpu=True)

In [ ]:
check_resource_usage()

In [ ]:
run_benchmark("gpu_sequential", gpu=True)

In [ ]:
stop_server()

---
## 9. Config 3 — GPU Batch (infrastructure-level: AMD MI100)

In [ ]:
start_server("gpu_batch", gpu=True)

In [ ]:
check_resource_usage()

In [ ]:
run_benchmark("gpu_batch", gpu=True)

In [ ]:
stop_server()

---
## 10. Config 4 — GPU ONNX (model-level: ONNX runtime)

In [ ]:
start_server("gpu_onnx", gpu=True)

In [ ]:
check_resource_usage()

In [ ]:
run_benchmark("gpu_onnx", gpu=True)

In [ ]:
stop_server()

---
## 11. Cleanup — delete server when done

In [ ]:
# only run this when you are fully done with the lease
# s.delete()